# Test ROUGE

Notebook này dùng để test nhanh phần đánh giá ROUGE đã được triển khai trong `src/evaluate.py`.

Hiện tại project chưa có output thật từ mô hình, nên notebook tạm dùng:

- `summary`: bản tóm tắt chuẩn, dùng làm reference.
- `article`: candidate/prediction tạm thời, chỉ để kiểm tra pipeline ROUGE chạy đúng.

Khi có output mô hình thật, chỉ cần đổi `PREDICTION_COL` sang tên cột chứa summary do mô hình sinh ra, ví dụ `pred_summary` hoặc `qwen_summary`.

## 1. Setup và import

In [1]:
from pathlib import Path
import sys

# Khi chạy code bằng Python thường trên Windows, stdout có thể không dùng UTF-8.
# Dòng này giúp print tiếng Việt có dấu không bị lỗi encoding.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

import pandas as pd

# Trong notebook Jupyter thường có sẵn IPython.display.
# Fallback bên dưới giúp cell vẫn chạy được nếu collaborator chạy code bằng Python thường.
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

# Notebook có thể được chạy từ thư mục `notebooks/` hoặc từ project root.
# Ta chỉ dùng đường dẫn tương đối để collaborator không cần sửa theo đường dẫn máy cá nhân.
root_candidates = [Path("."), Path("..")]
PROJECT_ROOT = next(
    (
        path
        for path in root_candidates
        if (path / "src" / "evaluate.py").exists()
        and (path / "data" / "train.csv").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy project root bằng đường dẫn tương đối. "
        "Hãy chạy notebook từ thư mục project root hoặc từ thư mục notebooks/."
    )

# Thêm project root vào sys.path để import được module trong thư mục `src`.
# Ví dụ khi chạy từ `notebooks/`, PROJECT_ROOT sẽ là `..`.
project_root_text = str(PROJECT_ROOT)
if project_root_text not in sys.path:
    sys.path.insert(0, project_root_text)

# Dùng lại đúng logic ROUGE nội bộ của project, không dùng thư viện ROUGE ngoài.
from src.evaluate import RougeEvaluator, average_scores, evaluate_dataframe

print("Project root tương đối:", PROJECT_ROOT)

Project root tương đối: ..


## 2. Cấu hình đánh giá

In [ ]:
# File train đang có hai cột chính là `article` và `summary`.
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"

# `summary` là bản tóm tắt chuẩn, dùng làm reference.
REFERENCE_COL = "summary"

# Hiện chưa có output mô hình, nên tạm dùng `article` làm prediction/candidate.
# Sau này đổi giá trị này sang cột output mô hình, ví dụ: "pred_summary" hoặc "qwen_summary".
PREDICTION_COL = "article"

# LIMIT = 10 giúp test nhanh. Đổi thành None nếu muốn chạy toàn bộ train.csv.
LIMIT = 10

# LOWERCASE=True giúp giảm khác biệt do viết hoa/viết thường khi so khớp token.
LOWERCASE = True

# MAX_SKIP=4 là mặc định của interface evaluate cho ROUGE-S skip-bigram.
MAX_SKIP = 4

config = {
    "data_path": str(DATA_PATH),
    "reference_col": REFERENCE_COL,
    "prediction_col": PREDICTION_COL,
    "limit": LIMIT,
    "lowercase": LOWERCASE,
    "max_skip": MAX_SKIP,
}

config

{'data_path': '..\\data\\train.csv',
 'reference_col': 'summary',
 'prediction_col': 'article',
 'limit': 10,
 'lowercase': True,
 'max_skip': 4}

## 3. Đọc dữ liệu và kiểm tra cột

In [3]:
# Đọc dữ liệu train bằng pandas.
# Nếu sau này đánh giá output mô hình từ file khác, có thể đổi DATA_PATH sang file CSV đó.
df = pd.read_csv(DATA_PATH)

# Kiểm tra sớm để lỗi dễ hiểu hơn nếu đổi sai tên cột.
required_cols = {REFERENCE_COL, PREDICTION_COL}
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f"Thiếu cột trong dữ liệu: {missing_cols}")

# Giữ một bản dữ liệu dùng để đánh giá.
# LIMIT khác None thì chỉ lấy vài dòng đầu để chạy nhanh trong lúc test notebook.
eval_df = df.head(LIMIT).copy() if LIMIT is not None else df.copy()

print("Shape dữ liệu gốc:", df.shape)
print("Shape dữ liệu đánh giá:", eval_df.shape)
print("Các cột hiện có:", df.columns.tolist())

# Hiển thị sample để collaborator kiểm tra nhanh đúng cột reference/prediction chưa.
sample_cols = list(dict.fromkeys([PREDICTION_COL, REFERENCE_COL]))
display(eval_df[sample_cols].head(3))

Shape dữ liệu gốc: (10775, 2)
Shape dữ liệu đánh giá: (10, 2)
Các cột hiện có: ['article', 'summary']


,article,summary
0,Gần 20 sự kiện được tổ chức trên toàn thành ph...,Hà Nội tổ chức gần 20 sự kiện từ 19/4 đến 10/5...
1,"Được thành lập năm 1897 tại Đức, Kempinski Hot...",Kempinski Hotels là một thương hiệu nổi tiếng ...
2,"Ngoài di chuyển đến Tuần Châu bằng đường bộ, m...",Bài viết giới thiệu các hoạt động vui chơi giả...


## 4. Khởi tạo ROUGE evaluator

In [4]:
# RougeEvaluator kế thừa interface `evaluate` trong `src/interfaces.py`.
# Class này tính ROUGE-1, ROUGE-2, ROUGE-L và ROUGE-S theo logic trong `src/evaluate.py`.
evaluator = RougeEvaluator(
    lowercase=LOWERCASE,
    max_skip=MAX_SKIP,
)

print("Các metric sẽ tính:", evaluator.rouge_types)

Các metric sẽ tính: ('rouge1', 'rouge2', 'rougeL', 'rougeS')


## 5. Tính ROUGE từng dòng

In [5]:
# evaluate_dataframe tính điểm ROUGE cho từng cặp reference - prediction.
# Output là DataFrame, mỗi dòng tương ứng một sample trong eval_df.
row_scores = evaluate_dataframe(
    df=eval_df,
    reference_col=REFERENCE_COL,
    prediction_col=PREDICTION_COL,
    evaluator=evaluator,
)

# Kiểm tra đủ các cột score như mong đợi.
expected_score_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rouge2_precision", "rouge2_recall", "rouge2_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "rougeS_precision", "rougeS_recall", "rougeS_f1",
]
missing_score_cols = [col for col in expected_score_cols if col not in row_scores.columns]
if missing_score_cols:
    raise AssertionError(f"Thiếu cột ROUGE: {missing_score_cols}")

# Bảng này chỉ gồm điểm ROUGE từng dòng.
display(row_scores.head())

# Bảng này ghép thêm text gốc để dễ debug sample có điểm cao/thấp.
scored_df = pd.concat(
    [eval_df[sample_cols].reset_index(drop=True), row_scores.reset_index(drop=True)],
    axis=1,
)
display(scored_df.head(3))

,rouge1_precision,rouge1_recall,rouge1_f1,rouge2_precision,rouge2_recall,rouge2_f1,rougeL_precision,rougeL_recall,rougeL_f1,rougeS_precision,rougeS_recall,rougeS_f1
0,0.200501,0.898876,0.327869,0.110553,0.500000,0.181070,0.127820,0.573034,0.209016,0.073232,0.337209,0.120332
1,0.216802,0.975610,0.354767,0.177748,0.803681,0.291111,0.165312,0.743902,0.270510,0.148571,0.678261,0.243750
2,0.171484,0.855769,0.285714,0.094595,0.475728,0.157810,0.109827,0.548077,0.182986,0.063953,0.326733,0.106969
3,0.171154,0.898990,0.287561,0.104046,0.551020,0.175041,0.123077,0.646465,0.206785,0.081238,0.437500,0.137031
4,0.248826,0.990654,0.397749,0.202353,0.811321,0.323917,0.206573,0.822430,0.330206,0.171158,0.696154,0.274763


,article,summary,rouge1_precision,rouge1_recall,rouge1_f1,rouge2_precision,rouge2_recall,rouge2_f1,rougeL_precision,rougeL_recall,rougeL_f1,rougeS_precision,rougeS_recall,rougeS_f1
0,Gần 20 sự kiện được tổ chức trên toàn thành ph...,Hà Nội tổ chức gần 20 sự kiện từ 19/4 đến 10/5...,0.200501,0.898876,0.327869,0.110553,0.500000,0.181070,0.127820,0.573034,0.209016,0.073232,0.337209,0.120332
1,"Được thành lập năm 1897 tại Đức, Kempinski Hot...",Kempinski Hotels là một thương hiệu nổi tiếng ...,0.216802,0.975610,0.354767,0.177748,0.803681,0.291111,0.165312,0.743902,0.270510,0.148571,0.678261,0.243750
2,"Ngoài di chuyển đến Tuần Châu bằng đường bộ, m...",Bài viết giới thiệu các hoạt động vui chơi giả...,0.171484,0.855769,0.285714,0.094595,0.475728,0.157810,0.109827,0.548077,0.182986,0.063953,0.326733,0.106969


## 6. Tính điểm ROUGE trung bình

In [6]:
# score_batch trả về macro-average trực tiếp theo cấu trúc nested dict.
# Macro-average nghĩa là mỗi dòng dữ liệu có trọng số như nhau khi lấy trung bình.
references = eval_df[REFERENCE_COL].astype(str).tolist()
predictions = eval_df[PREDICTION_COL].astype(str).tolist()

batch_scores = evaluator.score_batch(
    references=references,
    predictions=predictions,
)

# Chuyển nested dict thành bảng để dễ đọc trong notebook.
macro_scores = pd.DataFrame(batch_scores).T[["precision", "recall", "f1"]]
display(macro_scores)

# average_scores lấy trung bình từ row_scores đã flatten.
# Cell này hữu ích nếu collaborator muốn xử lý hoặc lọc row_scores trước rồi mới lấy trung bình.
flat_macro_scores = average_scores(row_scores)
display(pd.DataFrame([flat_macro_scores]))

,precision,recall,f1
rouge1,0.191753,0.949942,0.317061
rouge2,0.146548,0.729202,0.242345
rougeL,0.156684,0.777437,0.259004
rougeS,0.123595,0.623810,0.204825


,rouge1_precision,rouge1_recall,rouge1_f1,rouge2_precision,rouge2_recall,rouge2_f1,rougeL_precision,rougeL_recall,rougeL_f1,rougeS_precision,rougeS_recall,rougeS_f1
0,0.191753,0.949942,0.317061,0.146548,0.729202,0.242345,0.156684,0.777437,0.259004,0.123595,0.62381,0.204825


## 7. Cách dùng khi có output mô hình thật

Khi đã có summary do mô hình sinh ra, collaborator chỉ cần chuẩn bị DataFrame có:

- cột reference, thường là `summary`
- cột prediction, ví dụ `pred_summary` hoặc `qwen_summary`

Sau đó đổi cấu hình ở đầu notebook:

```python
REFERENCE_COL = "summary"
PREDICTION_COL = "pred_summary"  # hoặc "qwen_summary"
```

Nếu output mô hình nằm ở file CSV khác, đổi thêm:

```python
DATA_PATH = PROJECT_ROOT / "data" / "ten_file_output.csv"
```

Lưu ý: không cần sửa `src/evaluate.py` khi đổi model. Notebook này chỉ thay input đưa vào evaluator.